# **DCE Analysis Tool (GPU Accelerated)**

This tool allows you to estimate **Conditional Logit (CL)** and **Mixed Logit (MXL)** models using your own data. 
It uses a high-performance GPU engine (based on PyTorch) to estimate models in seconds/minutes rather than hours.

**Instructions:**
1.  Run the **Setup & Install** cell below (first time only).
2.  Run the **Start Application** cell to launch the interface.
3.  Upload your data, configure variables, and click **Run Analysis**.

---

In [ ]:
# @title 1. Setup & Install (Run Once)
# @markdown Installs necessary libraries (Torch, Pandas, etc.).
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import torch
    import pandas as pd
    import ipywidgets
    from scipy.optimize import minimize
except ImportError:
    print("Installing dependencies (this may take a minute)...")
    install("torch")
    install("pandas")
    install("scipy")
    install("ipywidgets")
    install("tqdm")
    install("openpyxl")  # For Excel export if needed
    print("Done!")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import qmc
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import base64

# Check Device
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ GPU not detected. Running on CPU (will be slower).")


In [ ]:
# @title 2. Core Engine (Hidden Code)
# @markdown This cell contains the generic GPU Model Logic. Run it to define the classes.

# =============================================================================
# HALTON SEQUENCE
# =============================================================================
def generate_halton_draws(n_individuals, n_draws, n_dims, seed=123):
    """Generate Halton sequence draws, transformed to standard normal."""
    if n_dims == 0:
        return np.zeros((n_individuals, n_draws, 0))
    sampler = qmc.Halton(d=n_dims, scramble=True, seed=seed)
    samples = sampler.random(n=n_individuals * n_draws)
    from scipy.stats import norm
    draws = norm.ppf(np.clip(samples, 1e-10, 1 - 1e-10))
    draws = draws.reshape(n_individuals, n_draws, n_dims)
    return draws

# =============================================================================
# GENERIC MIXED LOGIT MODEL (GPU)
# =============================================================================
class GenericMixedLogit:
    def __init__(self, data_dict, model_config, n_draws=100, device=DEVICE):
        """
        data_dict contains:
          - X: Tensor (n_obs, n_alts, n_vars)
          - y: Tensor (n_obs,) indices of chosen alt (0 to n_alts-1)
          - panel_idx: Tensor (n_obs,) mapping obs to individual index
          - n_individuals: int
        model_config: List of dicts [{'name': 'price', 'idx': 0, 'type': 'fixed'/'normal'/'lognormal'}]
        """
        self.device = device
        self.n_draws = n_draws
        
        # Unpack Data
        self.X = data_dict['X'].to(device).float()
        self.y = data_dict['y'].to(device).long()
        self.panel_idx = data_dict['panel_idx'].to(device).long()
        self.n_individuals = data_dict['n_individuals']
        self.n_obs = self.X.shape[0]
        self.n_alts = self.X.shape[1]
        
        # Parse Config
        self.params = []
        self.rand_param_indices = []
        self.fixed_param_indices = []
        
        # We map the flat Theta vector to these params
        self.theta_mapping = []
        idx_counter = 0
        
        # For generating draws
        self.n_random_vars = 0
        
        for p in model_config:
            p_def = {
                'name': p['name'],
                'col_idx': p['idx'],
                'type': p['type'],
                'draw_idx': -1
            }
            
            if p['type'] == 'fixed':
                p_def['theta_indices'] = [idx_counter]
                self.theta_mapping.append({'name': p['name'], 'type': 'fixed'})
                idx_counter += 1
                
            elif p['type'] in ['normal', 'lognormal']:
                p_def['theta_indices'] = [idx_counter, idx_counter + 1] # mu, sigma
                self.theta_mapping.append({'name': f"mu_{p['name']}", 'type': 'mu'})
                self.theta_mapping.append({'name': f"sd_{p['name']}", 'type': 'sigma'})
                idx_counter += 2
                
                p_def['draw_idx'] = self.n_random_vars
                self.n_random_vars += 1
            
            self.params.append(p_def)
        
        self.n_params = idx_counter
        
        # Generate Draws
        if self.n_random_vars > 0:
            print(f"Generating {self.n_draws} draws for {self.n_random_vars} random variables...")
            draws_np = generate_halton_draws(self.n_individuals, self.n_draws, self.n_random_vars)
            self.draws = torch.tensor(draws_np, device=device, dtype=torch.float32)
        else:
            self.draws = None

    def unpack_theta(self, theta_t):
        # Construct Beta matrix: (n_individuals, n_draws, n_vars)
        # n_vars is shape of X's last dim
        n_vars = self.X.shape[2]
        
        # Prepare Base Beta for all inds/draws
        # Shape: (n_individuals, n_draws, n_vars)
        # If no random params, n_draws dim is effectively broadcasted as 1
        
        betas = torch.zeros(self.n_individuals, self.n_draws, n_vars, device=self.device)
        
        for p in self.params:
            col = p['col_idx']
            indices = p['theta_indices']
            
            if p['type'] == 'fixed':
                val = theta_t[indices[0]]
                betas[:, :, col] = val
                
            elif p['type'] == 'normal':
                mu = theta_t[indices[0]]
                sigma = theta_t[indices[1]]
                # draw shape: (n_ind, n_draws)
                z = self.draws[:, :, p['draw_idx']]
                betas[:, :, col] = mu + sigma * z
                
            elif p['type'] == 'lognormal':
                mu = theta_t[indices[0]]
                sigma = theta_t[indices[1]]
                z = self.draws[:, :, p['draw_idx']]
                # Usually lognormal is exp(mu + sigma*z). 
                # Sometimes people want -exp(...) for price/cost. 
                # For generic software, we stick to positive exp(...). User can flip sign of data if needed.
                betas[:, :, col] = torch.exp(mu + sigma * z)

        return betas

    def neg_log_likelihood(self, theta_np):
        theta_t = torch.tensor(theta_np, device=self.device, dtype=torch.float32)
        
        # 1. Get Betas: (N_ind, N_draws, N_vars)
        betas = self.unpack_theta(theta_t)
        
        # 2. Expand Betas to Observation Level: (N_obs, N_draws, N_vars)
        # self.panel_idx maps obs -> ind
        betas_obs = betas[self.panel_idx] 
        
        # 3. Compute Utilities: V = X * Beta
        # X: (N_obs, N_alts, N_vars)
        # Betas: (N_obs, N_draws, N_vars)
        # We need V: (N_obs, N_draws, N_alts)
        # 
        # Einsum: 
        # n=obs, a=alt, v=var, d=draw
        # X[n,a,v] * Beta[n,d,v] -> V[n,d,a]
        
        V = torch.einsum('nav,ndv->nda', self.X, betas_obs)
        
        # 4. Compute Probabilities (Softmax over alternatives)
        # V shape: (N_obs, N_draws, N_alts)
        
        # LogSumExp for numerical stability
        # V_max = V.max(dim=2, keepdim=True)[0]
        # log_sum_exp = (V - V_max).exp().sum(dim=2).log() + V_max.squeeze(2)
        log_sum_exp = torch.logsumexp(V, dim=2)
        
        # Pick chosen alternative utility
        # y: (N_obs,)
        # Expand y to (N_obs, N_draws, 1)
        y_expanded = self.y.unsqueeze(1).unsqueeze(2).expand(-1, self.n_draws, 1)
        V_chosen = torch.gather(V, 2, y_expanded).squeeze(2) # (N_obs, N_draws)
        
        log_prob = V_chosen - log_sum_exp # (N_obs, N_draws)
        
        # 5. Panel Average (if Mixed Logit)
        # Need to sum log_probs for each individual, then take mean over draws
        
        # Aggregate log_prob by individual: (N_ind, N_draws)
        log_prob_ind = torch.zeros(self.n_individuals, self.n_draws, device=self.device)
        log_prob_ind.index_add_(0, self.panel_idx, log_prob)
        
        # Average over draws (in probability space)
        # P_i = (1/R) * sum(exp(LL_ir))
        # log(P_i) = log(sum(exp(LL_ir))) - log(R)
        log_prob_ind_avg = torch.logsumexp(log_prob_ind, dim=1) - np.log(self.n_draws)
        
        # Total Log Likelihood
        neg_ll = -log_prob_ind_avg.sum()
        
        return neg_ll.item()

    def fit(self, verbose=True):
        theta0 = np.zeros(self.n_params) + 0.01 # Start small
        
        # Callback for progress
        def callback(xk):
            if verbose:
               print(".", end="")
        
        print("Starting Estimation...")
        res = minimize(self.neg_log_likelihood, theta0, method='L-BFGS-B', 
                       options={'disp': verbose, 'maxiter': 500}, callback=callback)
        
        return res


In [ ]:
# @title 3. Start Application
# @markdown Run this cell to show the User Interface.

class DCEApp:
    def __init__(self):
        self.data_dict_file = None
        self.resp_file = None
        self.cov_file = None
        
        self.df_merged = None
        self.columns = []
        
        self.setup_ui()
        
    def setup_ui(self):
        # 1. File Uploaders
        self.u_dict = widgets.FileUpload(accept='.csv', description='Data Dict')
        self.u_resp = widgets.FileUpload(accept='.csv', description='Responses')
        self.u_cov = widgets.FileUpload(accept='.csv', description='Covariates')
        
        self.btn_load = widgets.Button(description="Load & Merge Data", button_style='primary')
        self.btn_load.on_click(self.on_load_data)
        
        self.out_status = widgets.Output()
        
        # 2. Config Area (Hidden initially)
        self.config_box = widgets.VBox([])
        
        # Layout
        self.ui = widgets.VBox([
            widgets.HTML("<h2>Phase 1: Upload Data</h2>"),
            widgets.HBox([self.u_dict, self.u_resp, self.u_cov]),
            widgets.HTML("<i>Note: Upload 3 CSVs. The app expects common ID columns to merge them.</i>"),
            self.btn_load,
            self.out_status,
            widgets.HTML("<hr>"),
            self.config_box
        ])
        
        display(self.ui)
        
    def on_load_data(self, b):
        self.config_box.children = []
        with self.out_status:
            clear_output()
            if not (self.u_dict.value and self.u_resp.value and self.u_cov.value):
                print("❌ Please upload all 3 files first.")
                # return # Uncomment in production. For now allow demo mode if empty?
                # Check for DEMO mode
                self.load_demo_data()
                return
            
            # Process Uploads
            def read_file(upload_widget):
                # ipywidgets 7 vs 8 difference handling
                if hasattr(upload_widget, 'value') and len(upload_widget.value) > 0:
                     # value is a tuple or dict
                    f = upload_widget.value[0] if isinstance(upload_widget.value, tuple) else upload_widget.value
                    if isinstance(f, dict): # older ipywidgets
                         content = f['content']
                    else: # new
                         content = f.content.tobytes()
                    return pd.read_csv(io.BytesIO(content))
                return None

            try:
                df_dict = read_file(self.u_dict)
                df_resp = read_file(self.u_resp)
                df_cov = read_file(self.u_cov)
                
                print("Files Loaded. Merging...")
                self.process_data(df_dict, df_resp, df_cov)
                
            except Exception as e:
                print(f"❌ Error reading files: {e}")
    
    def load_demo_data(self):
        print("⚠️ No files uploaded. Generating DEMO DATA for testing...")
        # Generate minimal valid structure
        # 100 people, 10 tasks, 3 alts
        N, T, J = 100, 10, 3
        data = []
        for i in range(N):
            for t in range(T):
                # Random choice
                choice = np.random.randint(0, J)
                for j in range(J):
                    row = {
                        'ID': i, 'Task': t, 'Alt': j, 
                        'Choice': 1 if j==choice else 0,
                        'Price': np.random.rand(),
                        'Time': np.random.rand(),
                        'Comfort': np.random.randint(0, 2)
                    }
                    data.append(row)
        self.df_merged = pd.DataFrame(data)
        print(f"✅ Demo Data Created: {len(self.df_merged)} rows.")
        self.show_config_ui()

    def process_data(self, df_d, df_r, df_c):
        # This needs user-specific logic usually, but we assume:
        # Response: ID, Task, Choice
        # Design: Block/Task, Alt, Attributes
        # Covariates: ID, Atts
        # We'll try to just basic merge or assume user uploaded Long Format already if simplest
        # For this tool to be 'generic software' we might ask them to upload ONE long file or standard merge.
        # Let's assume for this Prototype they upload pre-merged or we join on ID.
        
        # Simplification: Merge everything into one Big DataFrame
        # If not possible, warn user.
        
        print("Merging logic placeholder... Assuming Response file has most data or they are mergeable on 'ID'.")
        # In real software, this needs a column mapper step.
        # For now, let's assume df_resp is the main Long Format file or we just use df_resp if others empty.
        
        self.df_merged = df_r # Fallback
        # Try merge if IDs exist
        if 'RID' in df_r.columns and 'RID' in df_c.columns:
             self.df_merged = pd.merge(df_r, df_c, on='RID', how='left')
        
        print(f"✅ Data Merged. Rows: {len(self.df_merged)}")
        self.show_config_ui()

    def show_config_ui(self):
        # Columns
        cols = list(self.df_merged.columns)
        
        self.dd_id = widgets.Dropdown(options=cols, description='ID Col:')
        self.dd_choice = widgets.Dropdown(options=cols, description='Choice Col:')
        self.dd_alt = widgets.Dropdown(options=cols, description='Alt/Task Col:')
        
        # Guess defaults
        for c in cols:
            if 'ID' in c.upper() or 'RID' in c.upper(): self.dd_id.value = c
            if 'CHOICE' in c.upper() or 'BEST' in c.upper(): self.dd_choice.value = c
            if 'ALT' in c.upper() or 'TASK' in c.upper(): self.dd_alt.value = c

        # Variable Config
        self.var_boxes = []
        self.var_container = widgets.VBox()
        
        self.btn_add_var = widgets.Button(description="Add Variable")
        self.btn_add_var.on_click(self.add_var_row)
        
        # Run Button
        self.btn_run = widgets.Button(description="RUN MODEL", button_style='success', layout=widgets.Layout(width='100%', height='50px'))
        self.btn_run.on_click(self.on_run_model)
        
        self.config_box.children = [
            widgets.HTML("<h2>Phase 2: Configure Model</h2>"),
            widgets.HBox([self.dd_id, self.dd_choice, self.dd_alt]),
            widgets.HTML("<h3>Variables</h3>"),
            self.var_container,
            self.btn_add_var,
            widgets.HTML("<hr>"),
            self.btn_run
        ]
        
        # Add default rows
        self.add_var_row(None)
        self.add_var_row(None)
    
    def add_var_row(self, b):
        cols = list(self.df_merged.columns)
        dd_col = widgets.Dropdown(options=cols, description='Column:')
        dd_type = widgets.Dropdown(options=['fixed', 'normal', 'lognormal'], value='fixed', description='Type:')
        
        row = widgets.HBox([dd_col, dd_type])
        self.var_boxes.append({'row': row, 'col': dd_col, 'type': dd_type})
        self.var_container.children = tuple([x['row'] for x in self.var_boxes])

    def on_run_model(self, b):
        with self.out_status:
            print("⏳ Preparing Data & GPU...")
            
            # 1. Prepare Tensors
            try:
                id_col = self.dd_id.value
                choice_col = self.dd_choice.value
                
                # Sort is critical for reshaping
                df = self.df_merged.sort_values(by=[id_col, self.dd_alt.value])
                
                # Unique IDs mapping
                unique_ids = df[id_col].unique()
                id_map = {uid: i for i, uid in enumerate(unique_ids)}
                panel_idx = torch.tensor([id_map[uid] for uid in df[id_col]], dtype=torch.long)
                
                # Y target
                # Assuming choice is 0/1 or similar. We need index of chosen alt per task.
                # Wait, generic logit usually takes long format. 
                # If data is 1 row per alt: we need to reshape X to (N_tasks, N_alts, N_vars)
                # This is tricky without strict structure assumptions. 
                # Let's assume standard Long format: each task has J rows.
                
                n_alts = 3 # Auto-detect?
                # Detect alts by checking rows per ID/Task pair?
                # Simplification: Assume fixed J per task. 
                # Let's check the first task.
                first_id = df[id_col].iloc[0]
                # first_task = df[task_col].iloc[0]
                
                # For this generic tool, assume the user provides clean data where 
                # `len(df) % n_alts == 0` or similar.
                # Or we just reshape blindly based on counts.
                
                # Let's try to detect J
                # Count rows per person / count tasks?
                counts = df.groupby(id_col).size()
                # Assuming equal.
                
                # Just take the variables selected
                model_config = []
                x_list = []
                
                for i, vb in enumerate(self.var_boxes):
                    col_name = vb['col'].value
                    dist_type = vb['type'].value
                    
                    model_config.append({
                        'name': col_name,
                        'idx': i,
                        'type': dist_type
                    })
                    
                    vals = df[col_name].values.astype(np.float32)
                    x_list.append(vals)
                
                X_flat = np.stack(x_list, axis=1) # (TotalRows, N_Vars)
                
                # Reshape X to (N_Obs_Tasks, N_Alts, N_Vars)
                # We need to know N_Alts.
                # Hack: Assume 3 because user said "3 tables" (Pref1,2,3? or standard?)
                # Better: Ask user for N_Alts or infer.
                # We'll infer from "Alt" column unique values.
                n_alts = df[self.dd_alt.value].nunique()
                if n_alts < 2: n_alts = 3 # Fallback
                
                n_total_rows = len(df)
                n_obs_tasks = n_total_rows // n_alts
                
                X_tensor = torch.tensor(X_flat).reshape(n_obs_tasks, n_alts, len(x_list))
                
                # Recover One-Hot Choices
                # If choice col is 0/1, we need to find which index is 1.
                choice_flat = df[choice_col].values
                choice_reshaped = choice_flat.reshape(n_obs_tasks, n_alts)
                y_indices = torch.tensor(np.argmax(choice_reshaped, axis=1), dtype=torch.long)
                
                # Panel ID needs to be per task, not per row (since we collapsed rows)
                # Take every nth value
                panel_idx_task = panel_idx[::n_alts]
                
                print(f"🔹 Tensor Shape: {X_tensor.shape}. Alts: {n_alts}. Obs: {n_obs_tasks}.")
                
                # Run Model
                data_dict = {
                    'X': X_tensor, 
                    'y': y_indices, 
                    'panel_idx': panel_idx_task,
                    'n_individuals': len(unique_ids)
                }
                
                model = GenericMixedLogit(data_dict, model_config, n_draws=200)
                res = model.fit()
                
                print("✅ Done!")
                print("Likelihood:", -res.fun)
                print("Estimates:", res.x)
                
                # Formatted Output
                results_df = pd.DataFrame({
                    'Parameter': [m['name'] for m in model.theta_mapping],
                    'Estimate': res.x
                })
                display(results_df)
                results_df.to_csv("model_results.csv")
                print("Saved to model_results.csv")
                
            except Exception as e:
                print(f"❌ Runtime Error: {e}")
                import traceback
                traceback.print_exc()

app = DCEApp()